# 🧠 Doc2Graph — Interactive Pipeline Exploration

This notebook demonstrates each stage of the Doc2Graph classical NLP & ML knowledge graph extraction pipeline:
1. **Ingestion & Sentence Splitting** (`src.ingestion`)
2. **Named Entity Recognition (NER)** (`src.nlp.ner`)
3. **Rule-Based Coreference Resolution** (`src.nlp.coref_resolver`)
4. **Dependency-Pattern Relation Extraction** (`src.nlp.relation_extractor`)
5. **Predicate Normalization** (`src.nlp.predicate_normalizer`)
6. **Entity Resolution & Clustering** (`src.graph.entity_resolver`)

In [14]:
import os
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))

import spacy
from src.ingestion.preprocessor import preprocess, build_windows
from src.nlp.ner import NERRunner
from src.nlp.coref_resolver import RuleBasedCorefResolver
from src.nlp.relation_extractor import DepPatternExtractor
from src.nlp.predicate_normalizer import PredicateNormalizer
from src.nlp.relation_scorer import RelationScorer
from src.graph.entity_resolver import EntityResolver
from src.graph.graph_builder import build_resolved_entities, build_resolved_edges

## 1. Document Ingestion & Chunking

In [15]:
sample_doc = """Elon Musk founded Tesla in 2003. 
He also founded SpaceX to advance space exploration. 
Tesla is headquartered in Austin, Texas. 
Google acquired DeepMind in 2014. 
Demis Hassabis leads DeepMind in London."""

doc_bytes = sample_doc.encode('utf-8')
preprocessed = preprocess(doc_bytes, 'sample.txt')
chunks = preprocessed.chunks
print(f"Document ID: {preprocessed.doc_id[:12]}...")
print(f"Extracted {len(chunks)} sentence chunks:")
for c in chunks:
    print(f"  [{c.chunk_id[-4:]}] {c.text}")

Document ID: 71bc6b017087...
Extracted 5 sentence chunks:
  [f0:0] Elon Musk founded Tesla in 2003.
  [f0:1] He also founded SpaceX to advance space exploration.
  [f0:2] Tesla is headquartered in Austin, Texas.
  [f0:3] Google acquired DeepMind in 2014.
  [f0:4] Demis Hassabis leads DeepMind in London.


## 2. Named Entity Recognition (NER)

In [16]:
nlp = spacy.load("en_core_web_sm")
ner = NERRunner()

all_entities = []
entities_by_chunk = {}
for chunk in chunks:
    doc = nlp(chunk.text)
    ents = ner.run(doc, chunk.chunk_id, raw_text=chunk.text)
    entities_by_chunk[chunk.chunk_id] = ents
    all_entities.extend(ents)

print(f"Extracted {len(all_entities)} entities:")
for e in all_entities:
    print(f"  - {e.canonical_name} ({e.type}) [source: {e.source}]")

Extracted 10 entities:
  - Elon Musk (PERSON) [source: gazetteer]
  - Tesla (ORG) [source: gazetteer]
  - Tesla (ORG) [source: gazetteer]
  - Texas (GPE) [source: gazetteer]
  - Austin (GPE) [source: spacy]
  - Google (ORG) [source: gazetteer]
  - Deepmind (ORG) [source: spacy]
  - Demis Hassabis (PERSON) [source: gazetteer]
  - Deepmind (ORG) [source: gazetteer]
  - London (GPE) [source: gazetteer]


## 3. Coreference Resolution

In [17]:
coref = RuleBasedCorefResolver()
coref_map = coref.resolve(chunks, entities_by_chunk)
entity_by_id = {e.entity_id: e for e in all_entities}

print("Resolved coreferences:")
for ref, target_id in coref_map.items():
    target_name = entity_by_id[target_id].canonical_name if target_id in entity_by_id else target_id
    print(f"  '{ref}' -> {target_name}")

Resolved coreferences:
  '71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:1:he' -> Elon Musk


## 4. Relation Extraction & Normalization

In [18]:
extractor = DepPatternExtractor()
normalizer = PredicateNormalizer()
scorer = RelationScorer()
windows = build_windows(chunks, window_size=3, stride=2)

raw_triples = []
for window_chunks, center_chunk in windows:
    window_text = " ".join(c.text for c in window_chunks)
    window_doc = nlp(window_text)
    window_entities = [e for wc in window_chunks for e in entities_by_chunk.get(wc.chunk_id, [])]
    
    triples = extractor.extract(
        doc=window_doc,
        entities=window_entities,
        center_chunk_id=center_chunk.chunk_id,
        center_sent_idx=center_chunk.sent_idx,
        coref_map=coref_map,
        entity_by_id=entity_by_id,
    )
    raw_triples.extend(triples)

normalized = normalizer.normalize_batch(raw_triples)
scores = scorer.score_batch(raw_triples)

print(f"Extracted & Normalized {len(normalized)} Triples:")
for nt, s in zip(normalized, scores):
    print(f"  ({nt.subject.canonical_name}) -[:{nt.relation_type}]-> ({nt.object.canonical_name}) [conf: {s:.2f}]")

Extracted & Normalized 4 Triples:
  (Elon Musk) -[:FOUNDED]-> (Tesla) [conf: 1.00]
  (Austin) -[:RELATED_TO]-> (Texas) [conf: 0.97]
  (Deepmind) -[:RELATED_TO]-> (Austin) [conf: 0.03]
  (Demis Hassabis) -[:RELATED_TO]-> (Deepmind) [conf: 1.00]


## 5. Entity Resolution

In [19]:
resolver = EntityResolver(threshold=0.85)
id_map = resolver.resolve_within_doc(all_entities)

resolved_entities, entity_to_chunks = build_resolved_entities(all_entities, id_map)
resolved_edges = build_resolved_edges(normalized, id_map)

print(f"Final Graph Nodes: {len(resolved_entities)}")
for rent in resolved_entities:
    print(f"  Node: {rent.name} ({rent.type}) | Aliases: {rent.aliases}")

print(f"\nFinal Graph Edges: {len(resolved_edges)}")
for redge in resolved_edges:
    print(f"  Edge: {redge.source_id} -[:{redge.relation_type}]-> {redge.target_id}")

Final Graph Nodes: 8
  Node: Elon Musk (PERSON) | Aliases: ['Elon Musk']
  Node: Tesla (ORG) | Aliases: ['Tesla']
  Node: Texas (GPE) | Aliases: ['Texas']
  Node: Austin (GPE) | Aliases: ['Austin']
  Node: Google (ORG) | Aliases: ['Google']
  Node: Deepmind (ORG) | Aliases: ['DeepMind']
  Node: Demis Hassabis (PERSON) | Aliases: ['Demis Hassabis']
  Node: London (GPE) | Aliases: ['London']

Final Graph Edges: 4
  Edge: 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:0:gaz:Elon Musk -[:FOUNDED]-> 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:0:gaz:Tesla
  Edge: 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:2:26:32 -[:RELATED_TO]-> 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:2:gaz:Texas
  Edge: 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:3:16:24 -[:RELATED_TO]-> 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750ae3efac5991f0:2:26:32
  Edge: 71bc6b0170876a6686657684d6e5a541e8237070f7ff5617750